In [31]:
import os

In [32]:
%pwd

'c:\\Users\\USER'

In [33]:
if os.path.basename(os.getcwd()) == "research":
    os.chdir("../")
elif not os.path.exists("config/config.yaml"):
    import os
    target = os.path.abspath(r"c:\Users\USER\Downloads\Kidney_diesease _classification")
    if os.path.exists(os.path.join(target, "config/config.yaml")):
        os.chdir(target)

In [34]:
%pwd

'c:\\Users\\USER\\Downloads\\Kidney_diesease _classification'

In [35]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [36]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml,create_directories

In [37]:
class ConfigurationManager:
    def __init__(
            self,
            config_filepath=CONFIG_FILE_PATH,
            params_filepath=PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )

        return data_ingestion_config

In [38]:
import os
import urllib.request as request
import zipfile
import gdown
from cnnClassifier import logger
from cnnClassifier.utils.common import get_size

In [39]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self) -> str:
        '''
        Fetch data from the url
        '''
        try:
            dataset_url = self.config.source_URL
            zip_download_dir = self.config.local_data_file
            os.makedirs("artifacts/data_ingestion", exist_ok=True)
            logger.info(f"Downloading data from {dataset_url} into file {zip_download_dir}")

            file_id = dataset_url.split("/")[-2]
            prefix = 'https://drive.google.com/uc?export=download&id='

            # Retry up to 3 times for unreliable network connections
            max_retries = 3
            for attempt in range(1, max_retries + 1):
                try:
                    logger.info(f"Download attempt {attempt}/{max_retries}")
                    gdown.download(prefix + file_id, str(zip_download_dir), quiet=False)

                    if os.path.exists(zip_download_dir) and os.path.getsize(zip_download_dir) > 0:
                        logger.info(f"Downloaded data from {dataset_url} into file {zip_download_dir}")
                        return
                    else:
                        logger.warning(f"Attempt {attempt}: Download file missing or empty")
                except Exception as download_error:
                    logger.warning(f"Attempt {attempt} failed: {download_error}")
                    for f in os.listdir("artifacts/data_ingestion"):
                        if f.endswith(".part"):
                            os.remove(os.path.join("artifacts/data_ingestion", f))
                    if attempt == max_retries:
                        raise download_error

        except Exception as e:
            raise e

    def extract_zip_file(self):
        '''
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        '''
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [40]:
try:
    config = ConfigurationManager()

    data_ingestion_config = config.get_data_ingestion_config()

    data_ingestion = DataIngestion(config=data_ingestion_config)

    data_ingestion.download_file()
    data_ingestion.extract_zip_file()

except Exception as e:
    raise e

[2026-09-28 22:43:17,204: INFO: common]: yaml file: config\config.yaml loaded successfully
[2026-09-28 22:43:17,208: INFO: common]: yaml file: params.yaml loaded successfully
[2026-09-28 22:43:17,210: INFO: common]: Directory created at: artifacts
[2026-09-28 22:43:17,211: INFO: common]: Directory created at: artifacts/data_ingestion
[2026-09-28 22:43:17,213: INFO: 3182472182]: Downloading data from https://drive.google.com/file/d/1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3/view?usp=sharing into file artifacts/data_ingestion/data.zip
[2026-09-28 22:43:17,213: INFO: 3182472182]: Download attempt 1/3


Downloading...
From (original): https://drive.google.com/uc?id=1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3
From (redirected): https://drive.google.com/uc?id=1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3&confirm=t&uuid=169e0eec-d903-43f5-89a5-2249fe209fa9
To: c:\Users\USER\Downloads\Kidney_diesease _classification\artifacts\data_ingestion\data.zip
100%|██████████| 57.7M/57.7M [00:54<00:00, 1.05MB/s]

[2026-09-28 22:44:16,579: INFO: 3182472182]: Downloaded data from https://drive.google.com/file/d/1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3/view?usp=sharing into file artifacts/data_ingestion/data.zip
